# Исследовательский анализ данных (EDA) для кейса №5: Доставка еды

### Постановка задачи

Описание кейса: необходимо спрогнозировать точное количество
минут до завершения доставки с момента размещения заказа на
основе времени приготовления блюд, дистанции, категории пробок,
погоды и характеристик курьера.

Упростим описание:
Прикиньте как быстро курьер доставит заказ при условиях... и далее по списку

### Что ищем в датасете?

- Признаки, влияющие напрямую на время доставки курьера.
- Корреляции между принаками, слабые связи

### Цель конкретно данного ноутбука

Общий, широкий осмотр данного датасета

In [6]:
import pandas as pd

pd.set_option('display.max_columns', None)

df = pd.read_csv('Z:/Coding/URFU projects/Semester 1/Case 5 - Food delivery/Data/food_delivery_orders_dataset.csv')
print(f'Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов')

Размер датасета: 50000 строк, 37 столбцов


In [7]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.templates.default = 'plotly_white'

## Часть 2. Технический паспорт датасета

Смотрим структуру, типы, пропуски и диапазоны значений:

In [8]:
df.info()
df.describe(include='all').T

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 37 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   order_id                             50000 non-null  str    
 1   customer_id                          50000 non-null  str    
 2   restaurant_id                        50000 non-null  str    
 3   driver_id                            50000 non-null  str    
 4   order_timestamp                      50000 non-null  str    
 5   order_date                           50000 non-null  str    
 6   order_hour                           50000 non-null  int64  
 7   day_of_week                          50000 non-null  str    
 8   is_weekend                           50000 non-null  int64  
 9   city                                 50000 non-null  str    
 10  delivery_area                        50000 non-null  str    
 11  customer_age                         50

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
order_id,50000,50000,ORD_000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,50000,4979,CUS_00001,4510,NaN,NaN,NaN,NaN,NaN,NaN,NaN
restaurant_id,50000,250,RES_0001,8837,NaN,NaN,NaN,NaN,NaN,NaN,NaN
driver_id,50000,200,DRV_0066,315,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_timestamp,50000,25459,2025-10-19 13:00:00,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_date,50000,30,2025-10-23,1741,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_hour,50000.0,NaN,NaN,NaN,14.4659,5.231155,0.0,11.0,14.0,19.0,23.0
day_of_week,50000,7,Thursday,8472,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_weekend,50000.0,NaN,NaN,NaN,0.26712,0.44246,0.0,0.0,0.0,1.0,1.0
city,50000,4,City_A,19888,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
nulls = df.isna().sum()
print('Пропуски по столбцам (только ненулевые):')
print(nulls[nulls > 0].sort_values(ascending=False))
print()
print('Доля пропусков по статусу заказа:')
print(df.groupby('order_status').apply(lambda g: g.isna().mean().mean()).round(3))

Пропуски по столбцам (только ненулевые):
cancellation_reason             49132
tip_amount                        868
actual_delivery_time_minutes      868
late_delivery                     868
customer_rating                   868
dtype: int64

Доля пропусков по статусу заказа:
order_status
Cancelled    0.111
Completed    0.028
dtype: float64


**Выводы по пропускам:** все пропуски сосредоточены в полях, осмысленных только для завершённых доставок (`actual_delivery_time_minutes`, `late_delivery`, `customer_rating`) и `cancellation_reason` (только для отменённых). Проверяем гипотезу: пропуски **системные и объясняются статусом заказа**, а не потерей данных

In [10]:
print('Пропуски фактического времени по статусу:')
print(df.groupby('order_status')['actual_delivery_time_minutes'].apply(lambda s: s.isna().mean()).round(3))
print()
print('Пропуск reason у НЕ отменённых:', df.loc[df['order_status'] != 'Cancelled', 'cancellation_reason'].isna().mean())

Пропуски фактического времени по статусу:
order_status
Cancelled    1.0
Completed    0.0
Name: actual_delivery_time_minutes, dtype: float64

Пропуск reason у НЕ отменённых: 1.0


Гипотеза подтвердилась: у всех `Cancelled` нет времени доставки/оценки, у завершённых — нет причины отмены. Это не «грязь», а структура данных.

**Оптимизация типов:** категориальные строки переводим в `category`, даты — в `datetime`; рабочим датафреймом `d` дальше будут только **доставленные заказы** (таргет определён лишь для них):

In [11]:
cat_cols = ['city', 'delivery_area', 'customer_type', 'restaurant_type',
            'restaurant_primary_category', 'payment_method', 'weather', 'traffic_level',
            'day_of_week', 'order_status', 'cancellation_reason']
for c in cat_cols:
    df[c] = df[c].astype('category')
df['order_date'] = pd.to_datetime(df['order_date'])
df['order_timestamp'] = pd.to_datetime(df['order_timestamp'])

d = df[df['order_status'] == 'Completed'].copy()
print(f'Рабочий датафрейм d: {d.shape[0]} доставленных заказов из {df.shape[0]}')
print(f'Период данных: {df["order_date"].min():%d.%m.%Y} — {df["order_date"].max():%d.%m.%Y}')
print(f'Уникальных клиентов: {df["customer_id"].nunique()}, ресторанов: {df["restaurant_id"].nunique()}, курьеров: {df["driver_id"].nunique()}')

Рабочий датафрейм d: 49132 доставленных заказов из 50000
Период данных: 01.10.2025 — 30.10.2025
Уникальных клиентов: 4979, ресторанов: 250, курьеров: 200


## Часть 3. Качество данных: дубликаты, выбросы, правдоподобность

### 3.1 Дубликаты

In [12]:
print('Полных дубликатов строк:', df.duplicated().sum())
print('Дубликатов order_id:', df['order_id'].duplicated().sum())

Полных дубликатов строк: 0
Дубликатов order_id: 0


Дубликатов нет — заказы уникальны, каждая строка = один заказ. Хвала kaggle.

### 3.2 Числовые признаки: boxplots и выбросы

In [13]:
num_cols = ['actual_delivery_time_minutes', 'estimated_delivery_time_minutes',
            'restaurant_preparation_time_minutes', 'distance_km', 'items_count',
            'order_total', 'tip_amount']
fig = make_subplots(rows=2, cols=4, subplot_titles=num_cols + ['customer_rating'])
for i, col in enumerate(num_cols + ['customer_rating']):
    fig.add_trace(go.Box(y=d[col], name=col, showlegend=False,
                         marker_color='#636EFA' if i % 2 == 0 else '#EF553B'),
                  row=i//4+1, col=i%4+1)
fig.update_layout(height=650, title='Распределения числовых признаков', template='plotly_white')
fig.show()

print(d[num_cols].describe().T[['min', '50%', 'max']].round(1))

                                      min   50%    max
actual_delivery_time_minutes         10.0  33.0  172.0
estimated_delivery_time_minutes       8.0  28.0  128.0
restaurant_preparation_time_minutes   5.0  14.0   73.0
distance_km                           0.8   4.1   20.0
items_count                           1.0   2.0   10.0
order_total                           4.9  28.4  443.2
tip_amount                            0.0   0.0   53.6


**Выводы:**

- Время доставки: правый хвост до ~120 мин, но это физически возможные дальние доставки в шторм — **выбросы сохраняем** (они и есть «проблемные кейсы» кейса).
- `distance_km` до ~25 км, чаевые до значительных сумм — всё правдоподобно.
- Оценки клиентов дискретны (1–5), основная масса 4–5.

### 3.3 Проверка скорости курьеров (санити-чек)

In [14]:
d['transit_min'] = d['actual_delivery_time_minutes'] - d['restaurant_preparation_time_minutes']
d['speed_kmh'] = d['distance_km'] / d['transit_min'] * 60
print('Заказов с transit <= 0:', (d['transit_min'] <= 0).sum(),
      f'({(d["transit_min"] <= 0).mean():.2%})')
ok = d[d['transit_min'] > 0]
print(f'Медианная скорость курьера: {ok["speed_kmh"].median():.1f} км/ч, '
      f'95-й перцентиль: {ok["speed_kmh"].quantile(0.95):.1f} км/ч')
print(f'Средняя скорость по погоде:')
print(ok.groupby('weather', observed=True)['speed_kmh'].median().round(1))

Заказов с transit <= 0: 74 (0.15%)
Медианная скорость курьера: 14.7 км/ч, 95-й перцентиль: 25.3 км/ч
Средняя скорость по погоде:
weather
Clear     15.8
Cloudy    14.4
Rain      12.0
Storm      9.3
Name: speed_kmh, dtype: float64


Скорость медианно ~15 км/ч — реалистично для города (велосипед/скутер/пробки). В шторме скорость падает. Найдено 74 заказа (<0.2%) с transit ≤ 0 — вероятный артефакт генерации данных, для анализа скорости их исключаем, на выводы они не влияют.

## Часть 4. Одномерный анализ
### 4.1 Целевая переменная: фактическое vs ожидаемое время

In [15]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Фактическое время доставки',
                                                   'Прогноз платформы'))
fig.add_trace(go.Histogram(x=d['actual_delivery_time_minutes'], nbinsx=60, marker_color='#636EFA',
                           name='факт'), row=1, col=1)
fig.add_trace(go.Histogram(x=d['estimated_delivery_time_minutes'], nbinsx=60, marker_color='#EF553B',
                           name='прогноз'), row=1, col=2)
fig.update_layout(height=450, title='Распределение времени доставки (мин)', template='plotly_white',
                  showlegend=False, bargap=0.02)
fig.show()

err = d['estimated_delivery_time_minutes'] - d['actual_delivery_time_minutes']
print(f'Средняя ошибка прогноза: {err.mean():+.2f} мин (прогноз в среднем короче факта)')
print(f'Медианная ошибка: {err.median():+.2f} мин')
print(f'Доля опозданий среди доставленных: {d["late_delivery"].mean():.1%}')

Средняя ошибка прогноза: -4.23 мин (прогноз в среднем короче факта)
Медианная ошибка: -4.00 мин
Доля опозданий среди доставленных: 36.6%


**Вывод:** оба распределения скошены вправо (долгие доставки редки, но протяжённы). Прогноз платформы **систематически оптимистичен на ~4 минуты**, и это напрямую конвертируется в 37% опозданий — главный операционный риск кейса.

### 4.2 Баланс категориальных признаков

In [16]:
cat_cols = ['city', 'weather', 'traffic_level', 'payment_method']
fig = make_subplots(rows=2, cols=2, subplot_titles=cat_cols)
for i, col in enumerate(cat_cols):
    vc = df[col].value_counts()
    fig.add_trace(go.Bar(x=vc.index.astype(str), y=vc.values, showlegend=False,
                         marker_color=px.colors.qualitative.Set2[i]), row=i//2+1, col=i%2+1)
fig.update_layout(height=700, title='Баланс категориальных признаков', template='plotly_white')
fig.show()

- **Города несбалансированы**: City_A почти в 4 раза больше заказов, чем City_D. Учитывать при разрезах — маленький City_D даёт шумные оценки.
- **Погода смещена к безоблачной** (Clear 56%), штормов всего ~2.6% — выводы по Storm надо принимать осторожно.
- Наименьший дисбаланс у классов пробок
- В методах оплаты заметно выделяется кредитные карты 

### 4.3 Суточная и недельная динамика заказов

In [17]:
hour_counts = df.groupby('order_hour', observed=True).size().reset_index(name='orders')
fig = px.bar(hour_counts, x='order_hour', y='orders', template='plotly_white',
             title='Количество заказов по часам суток',
             labels={'order_hour': 'Час', 'orders': 'Заказов'})
fig.update_traces(marker_color='#636EFA')
fig.update_layout(height=420)
fig.show()

dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_counts = df.groupby('day_of_week', observed=True).size().reindex(dow_order).reset_index(name='orders')
fig2 = px.bar(dow_counts, x='day_of_week', y='orders', template='plotly_white',
              title='Количество заказов по дням недели', labels={'day_of_week': '', 'orders': 'Заказов'})
fig2.update_traces(marker_color='#AB63FA')
fig2.update_layout(height=420)
fig2.show()

**Выводы:**

- Два пика спроса: **обеденный (~12–13 ч)** и **вечерний (19–21 ч)** — классика доставки еды. Ночью (0–5 ч) спрос падает почти в 10 раз.
- **Аномалия по дням недели:** Thursday (~8.5k) и Wednesday (~8.2k) заметно «толще» остальных дней (~6.6k), при этом выходные не доминируют, как ожидалось по теории. 

Теория: Данные покрывают ровно один месяц (октябрь 2025), где часть дней недели просто встречается на 1 раз чаще. Всплески — артефакт календаря, а не поведенческая закономерность?

## Часть 5. Многомерный анализ и корреляции
### 5.1 Взаимосвязи с целевой переменной (scatter)

Берём случайную выборку 3000 заказов, чтобы графики оставались отзывчивыми:

In [18]:
sample = d.sample(3000, random_state=42)

fig = px.scatter(sample, x='distance_km', y='actual_delivery_time_minutes', color='weather',
                 opacity=0.7, template='plotly_white', height=500,
                 title='Расстояние vs фактическое время доставки (цвет — погода)',
                 labels={'distance_km': 'Расстояние, км',
                         'actual_delivery_time_minutes': 'Время доставки, мин'},
                 trendline='ols', trendline_scope='overall')
fig.show()

Заметен увеличивающийся разброс точек по мере увеличения расстояния доставки - "хвост кометы"

In [19]:
fig = px.scatter(sample, x='restaurant_preparation_time_minutes', y='actual_delivery_time_minutes',
                 color='items_count', color_continuous_scale='Viridis', opacity=0.7,
                 template='plotly_white', height=500,
                 title='Время готовки vs время доставки (цвет — число позиций)',
                 labels={'restaurant_preparation_time_minutes': 'Готовка, мин',
                         'actual_delivery_time_minutes': 'Доставка, мин'})
fig.show()

**Выводы:**

- `distance_km` — **сильнейший внешний драйвер** времени доставки (корреляция ~0.77): каждая пара километров добавляет заметное время, разброс растёт с расстоянием.
- Время готовки также сильно связано (r ~0.5) и растёт с числом позиций почти линейно: +1 позиция ≈ +2.5 мин готовки.
- Плохая погода визуально «поднимает» облако точек — проверим количественно в сегментном анализе.

### 5.2 Матрица корреляций phik

phik умеет нелинейные и категориальные связи. Для скорости считаем на выборке 10 000 строк; интервальные переменные указываем явно:

In [20]:
%pip install phik


Note: you may need to restart the kernel to use updated packages.


In [21]:
import phik

phik_cols = ['actual_delivery_time_minutes', 'estimated_delivery_time_minutes',
             'restaurant_preparation_time_minutes', 'distance_km', 'items_count',
             'order_total', 'order_hour', 'customer_age', 'delivery_partner_experience_months',
             'weather', 'traffic_level', 'city', 'restaurant_type', 'payment_method',
             'is_weekend', 'late_delivery', 'order_status']
interval_cols = ['actual_delivery_time_minutes', 'estimated_delivery_time_minutes',
                 'restaurant_preparation_time_minutes', 'distance_km', 'items_count',
                 'order_total', 'order_hour', 'customer_age', 'delivery_partner_experience_months']

phik_matrix = d.sample(10000, random_state=42)[phik_cols].phik_matrix(interval_cols=interval_cols)

z:\Coding\Anaconda\Lib\site-packages\phik\data_quality.py:82: UserWarning: Not enough unique value for variable order_status for analysis 1. Dropping this column
  warnings.warn(


In [22]:
fig = px.imshow(phik_matrix.round(2), text_auto=True, aspect='auto',
                color_continuous_scale='RdBu_r', zmin=0, zmax=1,
                title='Матрица корреляций phik (выборка 10k)')
fig.update_layout(height=750, template='plotly_white')
fig.show()

target_phik = phik_matrix['actual_delivery_time_minutes'].drop('actual_delivery_time_minutes').sort_values(ascending=False)
print('Топ связей с целевой переменной:')
print(target_phik.round(3))

Топ связей с целевой переменной:
estimated_delivery_time_minutes        0.918
distance_km                            0.790
restaurant_preparation_time_minutes    0.557
order_total                            0.531
late_delivery                          0.486
items_count                            0.415
traffic_level                          0.279
weather                                0.199
restaurant_type                        0.191
order_hour                             0.099
delivery_partner_experience_months     0.063
is_weekend                             0.055
customer_age                           0.041
city                                   0.019
payment_method                         0.000
Name: actual_delivery_time_minutes, dtype: float64


**Выводы:**

- Таргет почти полностью определяется **тремя составными частями пути**: расстоянием, временем готовки и самим прогнозом платформы (который, видимо, строится из этих же величин).
- `weather`, `traffic_level`, `city` дают **слабые, но ненулевые** связи — они объясняют не «сколько», а «насколько хуже прогноза» (разброс и опоздания), что увидим в Части 6.
- Опыт курьера, возраст клиента, тип оплаты практически независимы от времени доставки — на них при построении модели можно в целом не смотреть.
 
Примечательно, что estimated_delivery_time не имеет никаких связей с погодой на улице

## Часть 6. Сегментный анализ и выводы для бизнеса
### 6.1 Погода и пробки: время доставки и ошибка прогноза

In [23]:
weather_order = ['Clear', 'Cloudy', 'Rain', 'Storm']
seg = d.groupby('weather', observed=True).agg(
    n=('order_id', 'size'),
    late_rate=('late_delivery', 'mean'),
    actual=('actual_delivery_time_minutes', 'mean'),
    estimated=('estimated_delivery_time_minutes', 'mean')).reindex(weather_order)
seg['delay'] = (seg['actual'] - seg['estimated']).round(2)
print(seg.round(2))

fig = make_subplots(rows=1, cols=2, subplot_titles=('Фактическое vs прогноз (мин)', 'Доля опозданий'))
fig.add_trace(go.Bar(x=seg.index, y=seg['actual'], name='факт', marker_color='#EF553B'), row=1, col=1)
fig.add_trace(go.Bar(x=seg.index, y=seg['estimated'], name='прогноз', marker_color='#636EFA'), row=1, col=1)
fig.add_trace(go.Bar(x=seg.index, y=seg['late_rate'], name='доля опозданий', marker_color='#FFA15A'), row=1, col=2)
fig.update_layout(height=450, title='Влияние погоды', template='plotly_white', barmode='group')
fig.show()

             n  late_rate  actual  estimated  delay
weather                                            
Clear    27857       0.24   33.56      31.01   2.55
Cloudy   12581       0.39   35.12      30.83   4.29
Rain      7427       0.69   39.16      30.66   8.50
Storm     1267       0.89   47.07      31.42  15.65


In [24]:
traffic_order = ['Low', 'Medium', 'High']
segt = d.groupby('traffic_level', observed=True).agg(
    late_rate=('late_delivery', 'mean'),
    actual=('actual_delivery_time_minutes', 'mean'),
    estimated=('estimated_delivery_time_minutes', 'mean')).reindex(traffic_order)
segt['delay'] = (segt['actual'] - segt['estimated']).round(2)
print(segt.round(2))

fig = px.box(d, x='traffic_level', y='actual_delivery_time_minutes',
             category_orders={'traffic_level': traffic_order},
             color='traffic_level', template='plotly_white', height=450,
             title='Фактическое время доставки по уровню пробок')
fig.show()

               late_rate  actual  estimated  delay
traffic_level                                     
Low                 0.35   30.59      26.72   3.88
Medium              0.36   35.02      30.81   4.21
High                0.39   39.89      35.27   4.62


**Ключевая находка кейса:** прогноз платформы **практически не реагирует на погоду и пробки** — средний прогноз держится ~31 мин во всех погодных условиях, тогда как факт растёт с 33.6 (Clear) до 47.1 мин (Storm), а доля опозданий — **с 24% до 89%**! Пробки добавляют ~9 минут между Low и High. Прогноз злостно игнорирует внешние условия — главная, исправимая причина опозданий.

### 6.2 Ошибка прогноза растёт с расстоянием

In [25]:
d['dist_bin'] = pd.cut(d['distance_km'], [0, 2, 4, 6, 8, 10, 20],
                       labels=['0–2', '2–4', '4–6', '6–8', '8–10', '10+'])
segd = d.groupby('dist_bin', observed=True).agg(
    n=('order_id', 'size'),
    actual=('actual_delivery_time_minutes', 'mean'),
    estimated=('estimated_delivery_time_minutes', 'mean'),
    late_rate=('late_delivery', 'mean'))
segd['delay'] = (segd['actual'] - segd['estimated']).round(2)
print(segd.round(2))

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_bar(x=segd.index.astype(str), y=segd['late_rate'], name='доля опозданий', marker_color='#EF553B')
fig.add_scatter(x=segd.index.astype(str), y=segd['delay'], name='ошибка прогноза, мин',
                mode='lines+markers', secondary_y=True, line=dict(color='#636EFA', width=3))
fig.update_yaxes(title_text='Доля опозданий', secondary_y=False)
fig.update_yaxes(title_text='Прогноз минус факт, мин', secondary_y=True)
fig.update_layout(height=450, title='Чем дальше везти — тем хуже прогноз', template='plotly_white')
fig.show()

              n  actual  estimated  late_rate  delay
dist_bin                                            
0–2        5267   22.59      19.41       0.29   3.18
2–4       19149   28.41      24.77       0.33   3.64
4–6       13182   35.90      31.60       0.38   4.30
6–8        6198   43.83      38.86       0.42   4.98
8–10       2825   52.38      46.57       0.45   5.81
10+        2511   68.18      61.17       0.47   7.01


Опоздания монотонно растут с дистанцией: 29% на коротких плечах против 47% на 10+ км. Средняя ошибка прогноза растёт так же: короткие заказы прогнозируются почти идеально, длинные систематически недооцениваются на 6–7 минут.

### 6.3 Часы пик: растёт ли доля опозданий с нагрузкой?

In [26]:
segh = d.groupby('order_hour', observed=True).agg(
    n=('order_id', 'size'),
    late_rate=('late_delivery', 'mean'),
    actual=('actual_delivery_time_minutes', 'mean')).reset_index()
segh['is_peak'] = segh['n'] >= segh['n'].quantile(0.75)

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_bar(x=segh['order_hour'], y=segh['n'], name='заказов',
            marker_color=['#EF553B' if p else '#B6B6F5' for p in segh['is_peak']])
fig.add_scatter(x=segh['order_hour'], y=segh['late_rate'], name='доля опозданий',
                mode='lines+markers', line=dict(color='#636EFA', width=3), secondary_y=True)
fig.update_yaxes(title_text='Заказов', secondary_y=False)
fig.update_yaxes(title_text='Доля опозданий', secondary_y=True)
fig.update_layout(height=450, title='Спрос и опоздания по часам (красный — пиковые часы)',
                  template='plotly_white')
fig.show()

**Интересный «отрицательный результат», проверяющий теорию:** вопреки ожиданиям, **доля опозданий практически не растёт в часы пик** (держится ~36–37% весь день), хотя объём заказов отличается в разы. Видимо, платформа масштабирует прогноз под нагрузку: средний факт в пики чуть выше, но прогноз идёт с ним в ногу. Слепота прогноза к погоде/пробкам/дистанции — реальна, а к часу суток — нет.

### 6.4 Отмены: почему заказы не доходят до клиента

In [27]:
canc = df[df['order_status'] == 'Cancelled']['cancellation_reason'].value_counts()
print(canc)
print(f'Доля отмен: {(df["order_status"] == "Cancelled").mean():.2%}')

fig = px.pie(values=canc.values, names=canc.index, hole=0.45, template='plotly_white',
             title='Причины отмены заказов')
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=420)
fig.show()

cancellation_reason
Driver Unassigned         386
Restaurant Unavailable    160
Customer Cancelled        154
Changed Mind              153
Long Prep Delay            15
Name: count, dtype: int64
Доля отмен: 1.74%


Отмены редки (1.7%), но лидирует **"Driver Unassigned"** (~45% отмен) — заказы срываются из-за нехватки курьеров. Это дешёвое в исправление: резерв курьеров или приоритезация назначений в момент пика.

### 6.5 Портрет «проблемной» доставки: radar chart

Сравним средний профиль опоздавшего (late=1) и своевременного (late=0) заказа. Признаки нормируем на максимум для сопоставимости:

In [28]:
radar_feats = {'distance_km': 'Расстояние', 'restaurant_preparation_time_minutes': 'Готовка',
               'items_count': 'Позиций', 'order_total': 'Сумма заказа',
               'estimated_delivery_time_minutes': 'Прогноз', 'actual_delivery_time_minutes': 'Факт'}
prof = d.groupby('late_delivery', observed=True)[list(radar_feats)].mean()
prof_norm = prof / prof.max()

fig = go.Figure()
for late, label, color in [(0, 'Доставлен вовремя', '#00CC96'), (1, 'Опоздал', '#EF553B')]:
    fig.add_trace(go.Scatterpolar(r=prof_norm.loc[late].values, theta=list(radar_feats.values()),
                                  fill='toself', name=label, line=dict(color=color)))
fig.update_layout(height=500, title='Профиль опоздавшего vs своевременного заказа',
                  template='plotly_white', polar=dict(radialaxis=dict(range=[0, 1])))
fig.show()

print(prof.round(1))

               distance_km  restaurant_preparation_time_minutes  items_count  \
late_delivery                                                                  
0.0                    4.5                                 14.1          2.9   
1.0                    5.1                                 18.0          2.9   

               order_total  estimated_delivery_time_minutes  \
late_delivery                                                 
0.0                   38.0                             30.1   
1.0                   38.5                             32.4   

               actual_delivery_time_minutes  
late_delivery                                
0.0                                    31.1  
1.0                                    42.1  


Опоздавший заказ — в среднем **дальше, больше позиций и дольше готовится**. Но разрывы по этим признакам умеренные — решающий вклад дают внешние условия (погода/пробки) и накопленная ошибка прогноза, что согласуется с корреляционным анализом.

### 6.6 Проверка гипотезы «опоздание → плохая оценка клиента»

In [29]:
print('Средняя оценка клиента:')
print('  вовремя :', d.loc[d['late_delivery'] == 0, 'customer_rating'].mean().round(2))
print('  опоздал :', d.loc[d['late_delivery'] == 1, 'customer_rating'].mean().round(2))
print('Корреляция факт-время и оценки:',
      d['actual_delivery_time_minutes'].corr(d['customer_rating']).round(3))

fig = px.histogram(d, x='customer_rating', color='late_delivery', barmode='overlay', opacity=0.6,
                   nbins=32, template='plotly_white', height=420,
                   title='Оценка клиента: вовремя vs опоздание',
                   labels={'customer_rating': 'Оценка клиента', 'late_delivery': 'Опоздание (1=да)'})
fig.show()

Средняя оценка клиента:
  вовремя : 4.56
  опоздал : 3.83
Корреляция факт-время и оценки: -0.32


**Гипотеза подтверждается:** опоздание снижает среднюю оценку с **4.56 до 3.83** (−0.73 балла), корреляция времени доставки и оценки отрицательная (−0.32). Значит, борьба с опозданиями напрямую улучшает клиентский опыт — бизнес-эффект измерим.

## Итоги анализа

**Главные находки:**

1. **Прогноз платформы систематически занижен на ~4 минуты** (и хуже для дальних заказов, дождя и пробок)
2. **Прогноз слеп к внешним условиям.** Шторм: +13.5 мин к факту и 89% опозданий против 24% в ясную погоду — при этом прогноз одинаков (~31 мин). Пробки добавляют ~9 мин между Low и High.
3. **Дистанция — главный драйвер времени** (r ≈ 0.77), готовка — второй (r ≈ 0.5); каждая позиция в заказе ≈ +2.5 мин готовки.
4. **Опыт и рейтинг курьера не влияют** на время доставки — проблема не в людях, а в планировании.
5. **Часы пик не усугубляют опоздания** — платформа адаптирует прогноз ко времени суток, но не к погоде.
6. **Отмены (1.7%) в основном из-за нехватки курьеров** ("Driver Unassigned" — ~45% отмен).
7. **Опоздание стоит 0.73 балла** клиентской оценки — прямая связь операционки и опыта.
8. Всплеск заказов Wed–Thu — вероятный артефакт календаря месяца, а не поведение клиентов.
9. Ошибка прогноза увеличивается с дистанцией доставки заказа

**Ограничения анализа:** данные за 1 месяц; штормовых заказов мало (~2.6%) — оценки по Storm менее устойчивы; корреляции ≠ причинность (большая сумма заказа связана с долгой доставкой лишь через число позиций/готовку).